# 06 — ML Models: Predicting Engagement

Two models using PySpark MLlib:
1. Random Forest Regressor — predict post score.
2. Random Forest Classifier — predict high engagement (top 25% score).

### 1. Setup

In [ ]:
import sys
sys.path.insert(0, "..")

from utils import (
    get_spark, load_cricket_submissions,
    add_player_mentions, add_time_features, label_event_period,
    save_figure, save_pandas_to_local,
)

from pyspark.sql import functions as F
from pyspark.sql.functions import col, count, when
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import RegressionEvaluator, BinaryClassificationEvaluator
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

sns.set_theme(style="darkgrid")
plt.rcParams["figure.dpi"] = 120

### 2. Start Spark

In [ ]:
spark = get_spark("06_MLModels")
spark

### 3. Load and prepare data

In [ ]:
subs = load_cricket_submissions(spark)
subs = add_time_features(subs, ts_col="created_utc")
subs = label_event_period(subs, ts_col="created_dt")
subs = add_player_mentions(subs, text_col="title")
print("Data loaded and features added")

### 4. Select features and drop nulls

In [ ]:
feature_cols = ["hour_of_day", "day_of_week", "month_num",
                "mentions_dhoni", "mentions_kohli", "mentions_rohit",
                "subreddit", "event_period"]

ml_df = subs.select(feature_cols + ["score", "num_comments"]).dropna()

for flag in ["mentions_dhoni", "mentions_kohli", "mentions_rohit"]:
    ml_df = ml_df.withColumn(flag, col(flag).cast("int"))

print(f"ML dataset rows: {ml_df.count():,}")

### 5. Add high engagement label (top 25% score)

In [ ]:
p75 = ml_df.approxQuantile("score", [0.75], 0.01)[0]
print(f"75th percentile score: {p75}")

ml_df = ml_df.withColumn("high_engagement", when(col("score") >= p75, 1).otherwise(0))
print(ml_df.groupBy("high_engagement").count().toPandas())

### 6. Build pipeline — encode categoricals

In [ ]:
sub_indexer = StringIndexer(inputCol="subreddit",    outputCol="subreddit_idx",    handleInvalid="keep")
evt_indexer = StringIndexer(inputCol="event_period", outputCol="event_period_idx", handleInvalid="keep")
sub_ohe     = OneHotEncoder(inputCol="subreddit_idx",    outputCol="subreddit_ohe")
evt_ohe     = OneHotEncoder(inputCol="event_period_idx", outputCol="event_period_ohe")

numeric_features = ["hour_of_day", "day_of_week", "month_num",
                    "mentions_dhoni", "mentions_kohli", "mentions_rohit"]

assembler = VectorAssembler(
    inputCols=numeric_features + ["subreddit_ohe", "event_period_ohe"],
    outputCol="features",
)

### 7. Train/test split

In [ ]:
train_df, test_df = ml_df.randomSplit([0.8, 0.2], seed=42)
print(f"Train rows: {train_df.count():,}  |  Test rows: {test_df.count():,}")

### 8. Train Random Forest Regressor

In [ ]:
rf_reg = RandomForestRegressor(featuresCol="features", labelCol="score",
                                numTrees=50, maxDepth=6, seed=42)

reg_pipeline = Pipeline(stages=[sub_indexer, evt_indexer, sub_ohe, evt_ohe, assembler, rf_reg])
reg_model = reg_pipeline.fit(train_df)
print("Regression model trained")

### 9. Evaluate regression

In [ ]:
reg_preds = reg_model.transform(test_df)

rmse = RegressionEvaluator(labelCol="score", predictionCol="prediction", metricName="rmse").evaluate(reg_preds)
r2   = RegressionEvaluator(labelCol="score", predictionCol="prediction", metricName="r2").evaluate(reg_preds)

print(f"Regression — RMSE: {rmse:.2f}  |  R²: {r2:.4f}")
save_pandas_to_local(pd.DataFrame({"metric": ["RMSE", "R2"], "value": [rmse, r2]}), "06_regression_metrics.csv")

### 10. Train Random Forest Classifier

In [ ]:
rf_clf = RandomForestClassifier(featuresCol="features", labelCol="high_engagement",
                                 numTrees=50, maxDepth=6, seed=42)

clf_pipeline = Pipeline(stages=[sub_indexer, evt_indexer, sub_ohe, evt_ohe, assembler, rf_clf])
clf_model = clf_pipeline.fit(train_df)
print("Classification model trained")

### 11. Evaluate classification

In [ ]:
clf_preds = clf_model.transform(test_df)

auc = BinaryClassificationEvaluator(labelCol="high_engagement",
                                     rawPredictionCol="rawPrediction",
                                     metricName="areaUnderROC").evaluate(clf_preds)

correct  = clf_preds.filter(col("prediction") == col("high_engagement")).count()
accuracy = correct / clf_preds.count()

print(f"AUC-ROC: {auc:.4f}  |  Accuracy: {accuracy:.4f}")
save_pandas_to_local(pd.DataFrame({"metric": ["AUC-ROC", "Accuracy"], "value": [auc, accuracy]}),
                     "06_classification_metrics.csv")

### 12. Feature importances

In [ ]:
rf_stage    = clf_model.stages[-1]
importances = rf_stage.featureImportances.toArray()

n = len(numeric_features)
numeric_imp = importances[:n].tolist()
sub_imp     = float(importances[n:n + 200].sum())
evt_imp     = float(importances[n + 200:].sum())

imp_df = pd.DataFrame({
    "feature":    numeric_features + ["subreddit (OHE)", "event_period (OHE)"],
    "importance": numeric_imp + [sub_imp, evt_imp],
}).sort_values("importance", ascending=True)

print(imp_df)
save_pandas_to_local(imp_df, "06_feature_importances.csv")

### 13. Chart — feature importances

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(imp_df["feature"], imp_df["importance"], color="#2a9d8f", edgecolor="white")
ax.set_title("Feature Importances — RF Classifier (High Engagement)", fontsize=13)
ax.set_xlabel("Importance")
plt.tight_layout()
save_figure(fig, "06_feature_importances.png")
plt.show()

### 14. Summary table

In [ ]:
summary = pd.DataFrame({
    "Model":      ["Random Forest Regressor", "Random Forest Classifier"],
    "Task":       ["Predict post score", "Predict high engagement (top 25%)"],
    "Key Metric": [f"RMSE={rmse:.2f}, R²={r2:.4f}", f"AUC={auc:.4f}, Acc={accuracy:.4f}"],
})
print(summary.to_string(index=False))
save_pandas_to_local(summary, "06_model_summary.csv")

### 15. Stop Spark

In [ ]:
spark.stop()